In [4]:
!pip install dash jupyter-dash pandas yfinance plotly beautifulsoup4 pyngrok

In [ ]:
import dash
from dash import dcc, html, Input, Output
import plotly.graph_objects as go
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta

# --- 1. CONFIGURACIÓN DE LA APLICACIÓN ---
app = dash.Dash(__name__, external_stylesheets=['https://codepen.io/chriddyp/pen/bWLwgP.css'])
app.title = "Dashboard Técnico - Analista Bolivia"

# --- 2. BLOQUES DE CÓDIGO (Para mostrar en pantalla) ---
# Este es el código que se mostrará en el dashboard como referencia teórica/práctica
CODE_BLOCK_SCRAPING = """
# EJEMPLO DE CÓDIGO PARA SCRAPING LOCAL (BOLIVIA)
# (Este código es ilustrativo, requiere adaptación constante)
import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_bcv_data():
    # URL hipotética o real del BCB o BBV
    url = "https://www.bcb.gob.bo/?q=cotizaciones_tipo_cambio"
    try:
        # Nota: verify=False a veces es necesario para sitios gubernamentales
        response = requests.get(url, timeout=10, verify=False)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Lógica de scraping específica (clases, IDs)
        # tabla = soup.find('table', {'class': 'tabla-cotizaciones'})
        # df = pd.read_html(str(tabla))[0]

        return "Datos extraídos (Simulado)"
    except Exception as e:
        return f"Error: {e}"

# print(get_bcv_data())
"""

CODE_BLOCK_API = """
# CÓDIGO PARA API FINANCIERA GLOBAL (yfinance)
# Usado para el gráfico de velas a la derecha
import yfinance as yf

def get_global_data(ticker, period='1mo'):
    data = yf.download(ticker, period=period, interval='1d',threads=False,auto_adjust=True)
    return data

# data = get_global_data('SPY')
"""

# --- 3. DISEÑO (LAYOUT) DEL DASHBOARD ---
app.layout = html.Div(style={'backgroundColor': '#1e1e1e', 'color': '#white', 'padding': '20px'}, children=[

    # Encabezado
    html.H1("Dashboard Técnico: Integración Python para Análisis Financiero",
            style={'textAlign': 'center', 'color': '#00ff41', 'fontFamily': 'Courier New'}),
    html.P("Perspectiva para el Analista en Bolivia: Combinando Scraping Local y APIs Globales",
           style={'textAlign': 'center', 'color': '#cccccc'}),

    html.Hr(style={'borderColor': '#444'}),

    # Contenedor Principal (Flexbox)
    html.Div(style={'display': 'flex', 'flexDirection': 'row', 'gap': '20px'}, children=[

        # COLUMNA IZQUIERDA: Paneles de Código
        html.Div(style={'flex': '1', 'display': 'flex', 'flexDirection': 'column', 'gap': '20px'}, children=[

            # Panel de Control de Tickers
            html.Div(style={'backgroundColor': '#2d2d2d', 'padding': '15px', 'borderRadius': '5px'}, children=[
                html.H6("Controles de API Global", style={'color': '#00ff41'}),
                html.Label("Seleccionar Ticker (Proxy):", style={'color': '#ccc'}),
                dcc.Dropdown(
                    id='ticker-dropdown',
                    options=[
                        {'label': 'S&P 500 ETF (SPY)', 'value': 'SPY'},
                        {'label': 'Oro (GC=F)', 'value': 'GC=F'},
                        {'label': 'Petróleo WTI (CL=F)', 'value': 'CL=F'}
                    ],
                    value='SPY',
                    style={'color': '#000'}
                ),
            ]),

            # Panel Código Scraping
            html.Div(style={'backgroundColor': '#2d2d2d', 'padding': '15px', 'borderRadius': '5px', 'flex': '1'}, children=[
                html.H6("Lógica de Scraping Local (Bolivia)", style={'color': '#ffc107'}),
                html.Pre(CODE_BLOCK_SCRAPING, style={
                    'color': '#dcdcdc',
                    'backgroundColor': '#1a1a1a',
                    'padding': '10px',
                    'overflowX': 'auto',
                    'fontSize': '11px',
                    'border': '1px solid #444'
                })
            ]),

            # Panel Código API
            html.Div(style={'backgroundColor': '#2d2d2d', 'padding': '15px', 'borderRadius': '5px'}, children=[
                html.H6("Lógica de API Global (yfinance)", style={'color': '#007bff'}),
                html.Pre(CODE_BLOCK_API, style={
                    'color': '#dcdcdc',
                    'backgroundColor': '#1a1a1a',
                    'padding': '10px',
                    'overflowX': 'auto',
                    'fontSize': '11px',
                    'border': '1px solid #444'
                })
            ])
        ]),

        # COLUMNA DERECHA: Gráfico de Velas
        html.Div(style={'flex': '2', 'backgroundColor': '#2d2d2d', 'padding': '10px', 'borderRadius': '5px'}, children=[
            dcc.Graph(id='candlestick-graph', style={'height': '100%'})
        ])
    ])
])

# --- 4. INTERACTIVIDAD (CALLBACKS) ---


@app.callback(
    Output('candlestick-graph', 'figure'),   # ← ID corregido
    Input('ticker-dropdown', 'value')        # ← ID corregido
)
def update_candlestick(ticker):
    df = yf.download(ticker, period='3mo', interval='1d', progress=False)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [str(c).title() for c in df.columns]

    if df.empty or 'Close' not in df.columns:
        return go.Figure().update_layout(title="Esperando datos...")

    fig = go.Figure()
    fig.add_trace(go.Candlestick(
        x=df.index,
        open=df['Open'],
        high=df['High'],
        low=df['Low'],
        close=df['Close'],
        name=ticker
    ))
    fig.update_layout(
        title=f"Datos en tiempo real: {ticker}",
        xaxis_rangeslider_visible=False,
        yaxis=dict(autorange=True, fixedrange=False, title="Precio USD"),
        template="plotly_dark"
    )
    return fig

# --- 5. EJECUCIÓN ---
if __name__ == '__main__':
    # Ejecutar en modo debug, por defecto en http://127.0.0.1:8050
    #app.run(debug=True)
    from pyngrok import ngrok

    # Set your ngrok authtoken here. Replace 'YOUR_AUTHTOKEN' with your actual token.
    # You can get your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
    ngrok.set_auth_token("2xClm9FfRWPdeLiKSPWiuZWphk0_qwtHLqbs1LaYKSBYXfSG") # <-- Replace 'YOUR_AUTHTOKEN' with your actual ngrok authtoken

    # Abre un túnel público para el puerto 8050
    public_url = ngrok.connect(8050)
    print(f"Haz clic aquí para ver tu Dashboard: {public_url}")
    # Esto abrirá el dashboard directamente dentro de la celda de Colab
    app.run(port=8050)

Haz clic aquí para ver tu Dashboard: NgrokTunnel: "https://a22b-34-169-218-34.ngrok-free.app" -> "http://localhost:8050"
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 01:13:00] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 01:13:00] "GET /_dash-component-suites/dash/deps/react@18.v4_1_0m1774314714.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 01:13:00] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_1_0m1774314714.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 01:13:00] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_1_0m1774314714.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 01:13:00] "GET /_dash-component-suites/dash/dash_table/bundle.v7_1_0m1774314713.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 01:13:00] "GET /_dash-component-suites/dash/html/dash_html_components.v4_1_0m1774314714.min.js HTTP/

In [ ]:
import yfinance as yf
test_df = yf.download("SPY", period="1d")
print(test_df.columns)

/tmp/ipykernel_7388/2601765077.py:2: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed

MultiIndex([( 'Close', 'SPY'),
            (  'High', 'SPY'),
            (   'Low', 'SPY'),
            (  'Open', 'SPY'),
            ('Volume', 'SPY')],
           names=['Price', 'Ticker'])
